# 04 — Event Study

Computes Cumulative Abnormal Returns (CARs) for each ticker × event. This is the **core quantitative result** of the project.

**Prerequisite:** Run notebook 01 first.

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from utils import TICKERS, EVENT_DATES

## 4a. Run event study

In [ ]:
import event_study
event_study.run()

## 4b. Load results

In [ ]:
car_df      = pd.read_csv('../data/processed/car_results.csv')
car_summary = pd.read_csv('../data/processed/car_summary.csv')

# Format for display
display_cols = ['ticker', 'event', 'total_car', 't_stat', 'p_value', 'significant', 'beta']
print(car_summary[display_cols].to_string(index=False))

## 4c. CAR paths — all tickers by event

In [ ]:
events  = list(EVENT_DATES.keys())
tickers = list(TICKERS.keys())
colors  = ['#B5933A','#4A90D9','#7A9080','#D9534F','#9B59B6','#2ECC71']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for ax, event in zip(axes, events):
    ev_df = car_df[car_df['event'] == event]

    for ticker, color in zip(tickers, colors):
        t_df = ev_df[ev_df['ticker'] == ticker].sort_values('day')
        if t_df.empty:
            continue
        label = TICKERS.get(ticker, ticker)
        ax.plot(t_df['day'], t_df['car'] * 100, label=f'{label}', color=color, linewidth=1.8)

    ax.axvline(0, color='red', linestyle=':', linewidth=1.5, label='Event (t=0)')
    ax.axhline(0, color='black', linewidth=0.5, alpha=0.4)
    ax.set_title(event.replace('_', ' ').title())
    ax.set_xlabel('Trading Days Relative to Event')
    ax.set_ylabel('CAR (%)')
    ax.legend(fontsize=8, loc='upper left')

plt.suptitle('Cumulative Abnormal Returns by Event', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('../results/figures/car_paths.png', dpi=150, bbox_inches='tight')
plt.show()

## 4d. Summary heatmap — total CAR at end of event window

In [ ]:
pivot = car_summary.pivot(index='ticker', columns='event', values='total_car') * 100
pivot.index = [TICKERS.get(t, t) for t in pivot.index]

fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(pivot, annot=True, fmt='.2f', cmap='RdYlGn', center=0,
            linewidths=0.5, ax=ax, cbar_kws={'label': 'Total CAR (%)'})
ax.set_title('Total CAR (%) — Each Ticker × Event Window')
ax.set_xlabel('')
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.savefig('../results/figures/car_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 4e. Statistical significance

Stars: *** p<0.01  ** p<0.05  * p<0.10

In [ ]:
def stars(p):
    if p < 0.01: return '***'
    if p < 0.05: return '**'
    if p < 0.10: return '*'
    return ''

car_summary['stars']          = car_summary['p_value'].apply(stars)
car_summary['total_car_pct']  = (car_summary['total_car'] * 100).round(3)
car_summary['ticker_name']    = car_summary['ticker'].map(TICKERS)

print(car_summary[['ticker_name','event','total_car_pct','t_stat','p_value','stars']]
      .sort_values(['event','total_car_pct'], ascending=[True,False])
      .to_string(index=False))

car_summary.to_csv('../results/tables/car_summary_formatted.csv', index=False)